In [ ]:
from dotenv import load_dotenv 
import os
load_dotenv()


In [ ]:
from langchain_groq import ChatGroq     
api_key = os.getenv("GROQ_API_KEY")
llm=ChatGroq(model="llama-3.1-8b-instant",api_key=api_key)
llm.invoke("Hello World!")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [ ]:
speech="""Good morning everyone.

Today, I want to talk about how small changes can create a big impact. Many of us wait for the perfect time to start something new, but that moment rarely comes. Instead, progress begins when we take simple, consistent steps.

In our team, we started by improving just one process at a time. These small improvements reduced errors, saved time, and increased confidence. Over time, the results added up and transformed the way we work.

This experience taught us that success is not about dramatic change, but about steady effort. If each of us commits to small improvements every day, we can achieve meaningful and lasting growth.

Thank you. """

In [ ]:
chat_message =[SystemMessage(content="You are a expert in summarizing speeches."),
               HumanMessage(content=f"Summarize the following speech in 2-3 lines:\n Text:{speech}")]

In [ ]:
llm.invoke(chat_message).content

### Prompt Template for small content

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a expert in summarizing speeches.
            Summarize the following speech in 2-3 lines:
            Text: {speech} 
            Translate the summary into {language} language."""


prompt = ChatPromptTemplate.from_template(template)


In [ ]:
chain = prompt | llm
result = chain.invoke({
    "speech": speech, 
    "language": 'Spanish'
})
print(result.content)

### Stuff Document Chain for Summarization 

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("apjspeech.pdf")
doc=loader.load_and_split() 
doc[0].page_content
len(doc)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
template=""" Write a concise summary of the following speech: {text}

"""
prompt=ChatPromptTemplate.from_template(template)


In [ ]:
from langchain_classic.chains.summarize import load_summarize_chain

chain=load_summarize_chain(llm,chain_type="stuff",prompt=prompt,verbose=True)
chain.invoke({"input_documents":doc}) 


### Map Reduce Summarization 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
final_document=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100).split_documents(doc)

In [ ]:
len(final_document)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.summarize import load_summarize_chain

template=""" Write a concise summary of the following speech: {text} 
Summary: 
""" 
chunks_prompt=ChatPromptTemplate.from_template(template)

final_prompt=""" Provide the final summary of the entire speech with these import points.
Add a motivation Title,Start the precise summary with an introduction and provide the summary in number points for the speech.
Speech Points: {text} """ 
 
final_prompt= ChatPromptTemplate.from_template(final_prompt) # prompt to combine all the chunks summary. 

chain=load_summarize_chain(llm,chain_type="map_reduce",map_prompt=chunks_prompt,combine_prompt=final_prompt ,verbose=True) 
output=chain.invoke({"input_documents":final_document}) 

In [ ]:
chain=load_summarize_chain(llm,chain_type="refine",verbose=True) 

output=chain.invoke({"input_documents":final_document}) 




In [ ]:
!pip install numexpr